## How `df_web_features` was created

We used the Digital Footprints dataset as the basis for the web behavior features.

### 1. Loaded both web data parts
- Loaded `df_final_web_data_pt_1`
- Loaded `df_final_web_data_pt_2`
- Checked that both files have the same columns

### 2. Combined both parts
- Stacked both datasets into one full web dataset using `pd.concat()`
- Verified that the final row count matches the sum of both parts

### 3. Basic data quality checks
- Checked for duplicate rows
- Converted `date_time` to proper datetime format
- Confirmed that no timestamps became missing after conversion
- Checked that `process_step` only contains the expected values:
  - `start`
  - `step_1`
  - `step_2`
  - `step_3`
  - `confirm`

### 4. Sorted user events
- Sorted the dataset by:
  - `visitor_id`
  - `visit_id`
  - `date_time`
- This helps keep events in the correct order within each session.

### 5. Checked user journey patterns
- Looked at the sequence of `process_step` values per `visit_id`
- Found that most journeys follow the expected funnel logic, but some sessions include repeated or incomplete steps.
- We kept these patterns because they reflect real user behavior.

### 6. Aggregated to client level
- The raw web data is event-level, meaning one client can appear in multiple rows.
- For merging, we need one row per `client_id`.
- Therefore, we created step indicators per client:
  - Did the client reach `start`?
  - Did the client reach `step_1`?
  - Did the client reach `step_2`?
  - Did the client reach `step_3`?
  - Did the client reach `confirm`?

### 7. Final output
- Created `df_web_features`
- Structure:
  - one row per `client_id`
  - boolean columns for each funnel step
- Confirmed:
  - `120,157` unique clients
  - `0` duplicate `client_id`s

This dataset is now ready to merge with the experiment roster and client profile data.

In [1]:
import pandas as pd
import numpy as np

In [2]:
web_1_sample = pd.read_csv("../data/raw/df_final_web_data_pt_1.csv", nrows=1000)
web_2_sample = pd.read_csv("../data/raw/df_final_web_data_pt_2.csv", nrows=1000)

FileNotFoundError: [Errno 2] No such file or directory: '../data/raw/df_final_web_data_pt_1.csv'

In [ ]:
web_1_sample.head()

In [ ]:
web_2_sample.head()

In [ ]:
web_1_sample.shape, web_2_sample.shape

In [ ]:
web_1_sample.info()

In [ ]:
web_2_sample.info()

In [ ]:
web_1_sample.columns

In [ ]:
web_2_sample.columns

In [ ]:
web_1_sample.columns.equals(web_2_sample.columns)

In [3]:
web_1 = pd.read_csv("df_final_web_data_pt_1.csv")
web_2 = pd.read_csv("df_final_web_data_pt_2.csv")

df_web = pd.concat([web_1, web_2], ignore_index=True)

In [4]:
df_web.shape

(755405, 5)

In [ ]:
df_web.head()

In [ ]:
df_web.info()

In [ ]:
df_web["client_id"].nunique()

In [ ]:
df_web.groupby("client_id").size().describe()

In [ ]:
df_web["process_step"].value_counts()

In [ ]:
# 2. Duplicates
df_web.duplicated().sum()

In [5]:
# 3. Datetime check
df_web["date_time"] = pd.to_datetime(df_web["date_time"])
print(df_web["date_time"].dtype)

datetime64[us]


In [ ]:
df_web["process_step"].value_counts()

In [ ]:
df_web["process_step"].unique()

In [ ]:
df_web["date_time"].isna().sum()

In [6]:
df_web = df_web.sort_values(["client_id", "visit_id", "date_time"])

In [7]:
df_web["step_num"] = df_web["process_step"].map({"start": 0, "step_1": 1, "step_2": 2, "step_3": 3, "confirm": 4})

In [8]:
df_web['previous_step_num'] = df_web.groupby('visit_id')['step_num'].shift(1)

In [9]:
df_web["previous_step_num"].isna().sum()

np.int64(158095)

In [10]:
df_web["visit_id"].nunique()

158095

In [11]:
conditions = [
    df_web["previous_step_num"] > df_web["step_num"],
    df_web["previous_step_num"] < df_web["step_num"],
    df_web["previous_step_num"] == df_web["step_num"]
]

labels = ["backward", "forward", "repeat"]

df_web["step_direction"] = np.select(conditions, labels, default="first_visit")

In [16]:
df_web["time_on_each_step"] = (df_web["date_time"] - df_web.groupby("visit_id")["date_time"].shift(1)).dt.total_seconds()

In [18]:
df_web.to_csv("events_clean.csv", index=False)

## Initial Insights from Web Data

### 1. Sessions per Client
- Clients have multiple sessions on average.
- There is noticeable variability: some clients interact only once, while others return multiple times.
- This suggests different levels of engagement across users.

### 2. Funnel / Drop-off Behavior
- There is a clear drop-off at each step of the process:
  - Start → Step 1 → Step 2 → Step 3 → Confirm
- A significant portion of users does not reach the final confirmation step.
- This indicates potential friction points in the process that may need further investigation.

### 3. User Journey Patterns
- Most sessions follow the expected sequence (start → step progression → confirm).
- However, some irregular patterns exist (e.g. repeated steps or incomplete journeys).
- This suggests that not all users follow a clean linear path.

### 4. Data Structure Insight
- Raw web data is event-level (multiple rows per client).
- After aggregation, we successfully transformed it into a client-level dataset.
- Each client now has a clear behavioral profile (whether they reached each step).

---

## Note
These insights are based only on web interaction data.
They should be validated and extended after merging with experiment and client datasets.